# Notebook 09 — UDFs and Pandas UDFs

**Datasets:** `samples.nyctaxi.trips` · `samples.bakehouse.sales_transactions` · `samples.bakehouse.sales_customers`

**Difficulty:** Hard

**Topics covered:**
- Python UDF with `@F.udf`
- UDF with multiple input columns
- UDF for data masking
- `pandas_udf` scalar (vectorised UDF)
- `pandas_udf` for string normalisation
- Registering UDFs for use in `spark.sql`
- UDF vs built-in performance comparison
- Grouped map / `applyInPandas` pattern

> **Note:** Run cells top-to-bottom. All problems use the DataFrames loaded in the Setup cell.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.functions import pandas_udf
import pandas as pd

spark = SparkSession.builder.getOrCreate()

trips        = spark.table("samples.nyctaxi.trips")
transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")

# Register trips as a SQL temp view for problems that use spark.sql
trips.createOrReplaceTempView("trips")
transactions.createOrReplaceTempView("sales_transactions")

print("trips schema:");    trips.printSchema()
print("transactions schema:"); transactions.printSchema()
print("customers schema:");    customers.printSchema()

## Problem 1 — Basic Python UDF: fare category

Write a Python UDF that categorises `fare_amount` into:
- `'Short'`  when `fare_amount < 5`
- `'Medium'` when `5 <= fare_amount <= 20`
- `'Long'`   when `fare_amount > 20`

Apply it to `nyctaxi.trips`.

**Expected output columns:** `tpep_pickup_datetime`, `trip_distance`, `fare_amount`, `fare_category`



In [ ]:
# Problem 1 — write your solution here
# Assign result to: result_1
result_1 = None  # replace this


In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'tpep_pickup_datetime' in cols, "Missing: tpep_pickup_datetime"
assert 'trip_distance' in cols, "Missing: trip_distance"
assert 'fare_amount' in cols, "Missing: fare_amount"
assert 'fare_category' in cols, "Missing: fare_category"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
categories = {r['fare_category'] for r in result_1.select('fare_category').distinct().collect()}
assert categories.issubset({'Short', 'Medium', 'Long'}), f"Unexpected categories: {categories}"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — UDF with multiple inputs: formatted name

Write a UDF that takes `first_name` and `last_name` and returns `"LASTNAME, Firstname"` — last name upper-cased, first name title-cased.

Apply it to `sales_customers`.

**Expected output columns:** `customerID`, `formatted_name`, `country`



In [ ]:
# Problem 2 — write your solution here
# Assign result to: result_2
result_2 = None  # replace this


In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'formatted_name' in cols, "Missing: formatted_name"
assert 'country' in cols, "Missing: country"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
sample = result_2.select('formatted_name').limit(5).collect()
for row in sample:
    name = row['formatted_name']
    assert ',' in name, f"Expected 'LAST, First' format, got: {name}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — UDF for card number masking

Write a UDF that takes a `bigint` card number, converts it to a string, keeps only the **last 4 digits**, and replaces the rest with `'*'`.

Example: `1234567890123456` → `'************3456'`

Apply it to `sales_transactions`.

**Expected output columns:** `transactionID`, `cardNumber`, `masked_card`



In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3
result_3 = None  # replace this


In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'transactionid' in cols, "Missing: transactionID"
assert 'cardnumber' in cols, "Missing: cardNumber"
assert 'masked_card' in cols, "Missing: masked_card"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
sample = result_3.filter(F.col('masked_card').isNotNull()).select('masked_card').limit(5).collect()
for row in sample:
    m = row['masked_card']
    assert '*' in m, f"Expected '*' in masked card, got: {m}"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — `pandas_udf` (scalar): z-score of fare amount

Write a **scalar `pandas_udf`** that computes the z-score of `fare_amount` over the entire batch:

```
z = (x - mean(x)) / std(x)
```

Decorate with `@pandas_udf(returnType=T.DoubleType())`. Apply to `nyctaxi.trips`.

**Expected output columns:** `fare_amount`, `fare_zscore`



In [ ]:
# Problem 4 — write your solution here
# @pandas_udf(returnType=T.DoubleType())
# def zscore_udf(s: pd.Series) -> pd.Series: ...
# Assign result to: result_4
result_4 = None  # replace this


In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'fare_amount' in cols, "Missing: fare_amount"
assert 'fare_zscore' in cols, "Missing: fare_zscore"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
dt = dict(result_4.dtypes)
assert dt.get('fare_zscore') in ('double', 'float'), f"fare_zscore should be double, got {dt.get('fare_zscore')}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — `pandas_udf` for string normalisation

Write a `pandas_udf` that strips whitespace and title-cases the `product` column.

Apply it to `sales_transactions` and also compute the same result using the built-in `F.initcap(F.trim(...))` to compare.

**Expected output columns:** `product`, `normalised_product_udf`, `normalised_product_builtin`



In [ ]:
# Problem 5 — write your solution here
# Assign result to: result_5
result_5 = None  # replace this


In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'product' in cols, "Missing: product"
assert 'normalised_product_udf' in cols, "Missing: normalised_product_udf"
assert 'normalised_product_builtin' in cols, "Missing: normalised_product_builtin"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6 — Register UDF for SQL

Take the `fare_category` UDF from Problem 1 (or redefine it here). Register it with:

```python
spark.udf.register("fare_category_udf", udf_func, T.StringType())
```

Then use `spark.sql` to query it:

```sql
SELECT fare_amount, fare_category_udf(fare_amount) AS category
FROM trips
```

**Expected output columns:** `fare_amount`, `category`



In [ ]:
# Problem 6 — write your solution here
# then spark.sql("SELECT fare_amount, fare_category_udf(fare_amount) AS category FROM trips")
# Assign result to: result_6
result_6 = None  # replace this


In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'fare_amount' in cols, "Missing: fare_amount"
assert 'category' in cols, "Missing: category"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
categories = {r['category'] for r in result_6.select('category').distinct().collect()}
assert categories.issubset({'Short', 'Medium', 'Long'}), f"Unexpected categories: {categories}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7 — UDF performance comparison: built-ins win

For a simple operation — multiplying `fare_amount` by `1.1` to compute `fare_with_tip` — compare:

- **(a) Python UDF approach:** define a UDF that does `x * 1.1`
- **(b) Built-in approach:** `F.col('fare_amount') * 1.1`

Measure both with `.count()` to force evaluation (you can use `time.time()`).

Return the result using the **built-in approach** (column `fare_with_tip`). That is best practice.

**Expected output columns:** `fare_amount`, `fare_with_tip`

> **Why built-ins are preferred:** Python UDFs are executed row-by-row in the Python interpreter, bypassing Catalyst optimisations and incurring serialisation overhead between the JVM and Python. Built-in Spark functions are executed entirely in the JVM/native code and are fully optimisable by Catalyst. Use UDFs only when no built-in equivalent exists.

In [ ]:
# Problem 7 — write your solution here
# result_7 should use the BUILT-IN approach
# Assign result to: result_7
result_7 = None  # replace this


In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'fare_amount' in cols, "Missing: fare_amount"
assert 'fare_with_tip' in cols, "Missing: fare_with_tip"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8 — Grouped map with `applyInPandas`

> **Note:** `applyInPandas` is fully supported on Databricks Runtime. On open-source PySpark it requires the same Python version on all workers.

Normalise `fare_amount` to the **0–1 range within each `pickup_zip` group**:

```
normalised = (x - min(x)) / (max(x) - min(x))
```

Use `.groupBy("pickup_zip").applyInPandas(func, schema=...)` where `func` receives and returns a pandas DataFrame.

**Expected output columns:** `pickup_zip`, `fare_amount`, `normalised_fare`

```python
def normalise(pdf: pd.DataFrame) -> pd.DataFrame:
    lo, hi = pdf['fare_amount'].min(), pdf['fare_amount'].max()
    pdf['normalised_fare'] = (pdf['fare_amount'] - lo) / (hi - lo) if hi != lo else 0.0
    return pdf[['pickup_zip', 'fare_amount', 'normalised_fare']]

schema = "pickup_zip INT, fare_amount DOUBLE, normalised_fare DOUBLE"
result_8 = trips.groupBy("pickup_zip").applyInPandas(normalise, schema=schema)
```

In [ ]:
# Problem 8 — write your solution here
# Assign result to: result_8
result_8 = None  # replace this


In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'pickup_zip' in cols, "Missing: pickup_zip"
assert 'fare_amount' in cols, "Missing: fare_amount"
assert 'normalised_fare' in cols, "Missing: normalised_fare"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, "Got 0 rows"
# normalised_fare should be in [0, 1]
out_of_range = result_8.filter(
    (F.col('normalised_fare') < -0.001) | (F.col('normalised_fare') > 1.001)
).count()
assert out_of_range == 0, f"{out_of_range} rows have normalised_fare outside [0,1]"
print(f"Problem 8 passed ✓  ({cnt} rows)")